# 03 Ă‚Â· Dream-1 through the workbench

Use Dream's cloud policy with the repository's existing experimental single-arm
adapter. The physical SO101 is the right arm; the adapter supplies dummy left-arm
state and fills the absent left-wrist image from the overhead view.

**Kernel:** `Ramion Ă‚Â· Dream-1`  
**Credentials:** enter your own Dream API key in the hidden prompt when starting inference.  
**Cloud data:** inference sends both camera views and robot state to Dream Labs.

The pinned SDK can disable additional recording. The default below requests it off
and verifies that state before sending observations. You may explicitly set
`ALLOW_ADDITIONAL_RECORDING = True` to accept the SDK recorder. Disabling additional
recording does not prevent the images needed for inference from being transmitted.


**Physical validation required before another autonomous run.** A real run folded inward and stopped on a wrist-flex tracking error greater than 15 degrees. Execution has been disarmed. A successful inference preview does not validate this single-arm adaptation. Check measured 3D pose against the physical arm, joint zero/directions, gripper endpoints, and camera/frame correspondence before rearming. Do not invert actions or increase the tracking threshold to bypass this incident.


In [ ]:
import os
from pathlib import Path
from soarm_lab.notebooks.client import WorkbenchClient

REPO = next((p for base in (Path.cwd(), *Path.cwd().parents)
             for p in (base, base / 'apps' / 'arm-workbench')
             if (p / 'packages' / 'soarm').is_dir()), None)
if REPO is None:
    raise RuntimeError('Open this notebook from the Ramion checkout.')
WORKSPACE = REPO / 'workspace' / 'notebooks'
WORKSPACE.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('HF_HOME', str(REPO / 'workspace' / 'huggingface'))
client = WorkbenchClient(os.environ.get('RAMION_WORKBENCH_URL', 'http://100.85.56.14:8765'))
TASK = 'Move the stuffed toy mouse onto the left gray area.'


In [ ]:
import time
from soarm_lab.notebooks.credentials import prompt_dream_key
from soarm_lab.notebooks.models import DreamInference
from soarm_lab.notebooks.client import WorkbenchError

CHECKPOINT = 'dreamlabs/dream-1-beta'
PREVIEW_SECONDS = 120
MAX_OUTPUT_AGE_SECONDS = 15  # Model prediction age; local feedback must still be fresh.
ALLOW_ADDITIONAL_RECORDING = False  # Set True only to opt into the SDK recorder.
TASK = 'Move the stuffed toy mouse onto the left gray area.'
client.show(client.snapshot())


Standard beta is selected for this slower workflow. Preview waits up to 120 seconds and makes at most three requests, respecting provider backoff. Predictions may be up to 15 seconds old; keep the scene unchanged while waiting. Local camera/robot freshness checks and the workbench motion watchdog remain active. The live episode is now limited to 60 seconds; stop it in the workbench once the mouse is placed. Inspect a new preview before explicitly rearming.


## 1. Preview a small number of model requests

Set `PREVIEW_DREAM = True` to make the bounded cloud preview. The loop observes the
robot and displays proposed targets; it never claims control or sends an action.
The first request can take time while the remote model starts. Action chunks more
than the configured 15 seconds old relative to their originating observation are refused, so
a slow cloud response can end this smoke test. Do not silently relax that limit.
Before arming, the preview allows up to 21 snapshot attempts with 0.5-second pauses
after local model initialization; the live motion loop never retries stale feedback.


In [ ]:
def preview_snapshot():
    # Startup can briefly delay feedback. Retry only before any session is armed.
    for attempt in range(21):
        try:
            return client.snapshot()
        except WorkbenchError:
            if attempt == 20:
                raise
            time.sleep(0.5)

def wait_for_chunk(model, timeout_s=PREVIEW_SECONDS):
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        observation = preview_snapshot()
        result = model.step(observation.joints, observation.images, ages=observation.ages)
        if result.done:
            return result  # Show refused/late model output without permitting motion.
        if result.metadata.get('chunks', 0) > 0:
            return result
        time.sleep(0.05)
    return model.preview_timeout(timeout_s)  # Preserve provider errors and retry guidance.

PREVIEW_DREAM = False
preview = None
if PREVIEW_DREAM:
    model = DreamInference(
        prompt=TASK, checkpoint=CHECKPOINT, allow_recording=ALLOW_ADDITIONAL_RECORDING,
        api_key=prompt_dream_key(), max_requests=3, max_seconds=PREVIEW_SECONDS, max_output_age_s=MAX_OUTPUT_AGE_SECONDS,
    )
    try:
        observation = client.snapshot()
        model.start(observation.joints, observation.images, ages=observation.ages,
                    limits=observation.state['limits'])
        preview = wait_for_chunk(model)
        print({'elapsed_s': preview.elapsed_s, 'note': preview.note,
               'executable': preview.executable, 'metadata': preview.metadata})
        print('Executable proposed targets (degrees / gripper percent):', preview.actions[:3])
        if preview.raw_actions is not None:
            print('Raw Dream output for inspection only (arm radians, gripper 0..1):')
            print(preview.raw_actions[:3])
        if not preview.executable:
            print('No executable targets: inspect the note above; this result cannot arm motion.')
    finally:
        model.close()
else:
    print('Set PREVIEW_DREAM = True to make a bounded cloud preview with no motion.')


## 2. Run a short supervised episode

Inspect the preview and current scene before enabling motion. This creates a new
bounded Dream session, waits for its first chunk while the robot remains under the
workbench's current control, then explicitly claims control for at most five seconds.

The workbench enforces coordinated targets, limits, stale feedback checks, a command
watchdog and a session deadline. These controls do not establish obstacle clearance or
make the experimental model reliable. Physical testing is deferred to a session with
an operator beside the arm.


In [ ]:
RUN_SECONDS = 60  # Supervised episode; STOP remains available in the workbench.
MAX_RUN_REQUESTS = 45  # Includes warm-up and live inference; not a monetary cap.
EXECUTE = False
ARM_CONFIRMATION = ''  # Type ARM ROBOT only for the supervised run.
if EXECUTE:
    if preview is None or not preview.executable:
        raise RuntimeError('Run and inspect a successful inference preview first.')
    model = DreamInference(
        prompt=TASK, checkpoint=CHECKPOINT, allow_recording=ALLOW_ADDITIONAL_RECORDING,
        api_key=prompt_dream_key(), max_requests=MAX_RUN_REQUESTS, max_seconds=PREVIEW_SECONDS + RUN_SECONDS + 15, max_output_age_s=MAX_OUTPUT_AGE_SECONDS,
    )
    try:
        observation = client.snapshot()
        model.start(observation.joints, observation.images, ages=observation.ages,
                    limits=observation.state['limits'])
        first_chunk = wait_for_chunk(model)
        if first_chunk.done or not first_chunk.executable:
            raise RuntimeError('First chunk is not executable: ' + first_chunk.note)

        run_status = {'reason': 'Episode time limit reached', 'last_report': 0.0}

        def dream_step(observation):
            result = model.step(observation.joints, observation.images, ages=observation.ages,
                                command=observation.state['command'])
            if time.monotonic() - run_status['last_report'] >= 2:
                print('Dream:', result.note, '| requests:', result.metadata.get('requests'),
                      '| chunks:', result.metadata.get('chunks'), flush=True)
                run_status['last_report'] = time.monotonic()
            if result.done:
                run_status['reason'] = result.note
                raise StopIteration(result.note)
            return result.actions[0] if result.executable and result.actions else None

        records = client.run_policy(
            dream_step, hz=20, speed=8, seconds=RUN_SECONDS, confirmation=ARM_CONFIRMATION,
        )
        print('Stopped:', run_status['reason'])
        print('Accepted target updates:', len(records))
        print('Elapsed control time:', round(records[-1]['elapsed_s'], 1) if records else 0, 'seconds')
        if records:
            print('Measured joint motion (degrees; gripper in percentage points):')
            for joint in records[0]['measured']:
                values = [row['measured'][joint] for row in records]
                print(f'  {joint}: {values[0]:.2f} -> {values[-1]:.2f}; '
                      f'net {values[-1] - values[0]:+.2f}, range {max(values) - min(values):.2f}')
            print('This reports servo feedback, not successful completion of the mouse task.')
    finally:
        model.close()
    client.show(client.snapshot())
else:
    print('Motion disabled. Review the preview, then explicitly arm only during the supervised session.')


## Explicit STOP

Use the workbench's STOP button or deliberately run the request below. Interrupting
the episode cell also attempts release; the workbench has an independent watchdog.


In [ ]:
REQUEST_STOP = False
if REQUEST_STOP:
    client.stop()
    print('STOP requested.')


## Interpreting results

Report whether the toy was grasped, lifted, placed and released, and which gray region
it reached. Save useful observations locally if desired, without keys or controller
tokens. A successful API response or finite action chunk is not proof of task success.

Verification on 2026-11-04 obtained a real Flash `48 Ä‚â€” 6` chunk in 1.851 seconds,
with source-observation age 1.891 seconds and additional recording off. It was
accepted for preview; no robot action was sent. A subsequent 3.489-second reply
was rejected by the previous 3-second age limit. A warm kernel did not guarantee
a fresh reply, so inspect each result and keep the freshness gate.

The provider initially returned `DA-2000 NO_CAPACITY`, reason `pool_warming`, then
recovered after an explicitly bounded retry. If it recurs, inspect
`preview.metadata["last_error"]`, wait the reported retry interval, and explicitly
rerun the bounded preview. Do not loop indefinitely or arm without an executable result.

For checkpoint access errors, inspect the account's models before changing the ID.
For camera/state errors, fix the workbench input rather than bypassing freshness checks.
The extra recorder is disabled unless explicitly enabled; request/time limits bound these smoke tests but
are not monetary billing caps.

The slower standard-beta configuration was subsequently verified: a valid chunk arrived in 5.64 seconds (source age 5.70 seconds) and passed the new 15-second limit. No robot commands were sent.
